# FuelWise AI — entrenamiento en Google Colab

Este notebook ejecuta el pipeline completo (descarga de VED → preprocesamiento → LSTM/GRU → evaluación → modelo de valor) y descarga los artefactos para usar la app localmente.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`.
La GPU es opcional: en CPU el entrenamiento tarda ~15–20 min; con T4 ~3–5 min. El preprocesamiento (~5–8 min) usa CPU en ambos casos.

In [ ]:
!nvidia-smi -L || echo 'Sin GPU: se usará CPU'

## 1. Subir el proyecto
Opción A: sube `fuelwise.zip` (el que te entregamos). Opción B: clona tu repositorio de GitHub.

In [ ]:
# Opción A: subir el zip
from google.colab import files
up = files.upload()            # selecciona fuelwise.zip
!unzip -q -o fuelwise.zip

# Opción B (comenta lo anterior y usa esto si ya está en GitHub):
# !git clone https://github.com/<usuario>/<repositorio>.git fuelwise

In [ ]:
%cd /content/fuelwise
!pip install -q py7zr streamlit plotly openpyxl pyarrow

## 2. Pipeline completo
Descarga ~176 MB de GitHub (gsoh/VED), descomprime ~3 GB de CSV y entrena ambos modelos.

In [ ]:
!python run_all.py

## 3. Resultados

In [ ]:
import json, pandas as pd
m = json.load(open('artifacts/metrics.json'))
display(pd.DataFrame(m['block']['test']).T.round(3))
display(pd.DataFrame(m['trip']['test']).T.round(3))
v = json.load(open('artifacts/value_results.json'))
display(pd.DataFrame({k: x['test'] for k, x in v['methods'].items()}).T.round(3))
display(pd.read_csv('artifacts/iterations.csv'))

## 4. (Opcional) Iteración 1: modelo directo, para la tabla de iteraciones
Entrena prediciendo el consumo directamente (sin corrección sobre persistencia). Al terminar vuelve a entrenar el modelo residual final.

In [ ]:
# !python -m src.train --target direct && python -m src.evaluate
# !python -m src.train --target residual && python -m src.evaluate && python -m src.value_model

## 5. Descargar artefactos
Descomprime `artifacts.zip` dentro de la carpeta `fuelwise/` de tu computadora (reemplaza `artifacts/`).

In [ ]:
!zip -q -r artifacts.zip artifacts
from google.colab import files
files.download('artifacts.zip')

## 6. (Opcional) Abrir la app desde Colab
Crea un enlace público temporal con Cloudflare Tunnel. Para la feria es más estable correrla localmente.

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nohup streamlit run app/app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!sleep 5 && ./cloudflared tunnel --url http://localhost:8501 2>&1 | grep -m1 -o 'https://.*trycloudflare.com'